In [1]:
import os
import json
import random
import uuid
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta
from tqdm import tqdm

In [19]:
NUM_STORES = 800
DAYS = 7
MAX_DAILY_SALES = 200000
SHIPMENTS_PER_DAY = 50000
INVENTORY_PRODUCTS_PER_WH = 800

START_DATE = datetime(2026, 3, 10)

DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f"{DATA_DIR}/warehouse_inventory", exist_ok=True)
os.makedirs(f"{DATA_DIR}/shipments", exist_ok=True)

In [14]:
STATES_CITIES = {
"California":["Los Angeles","San Diego","San Jose","San Francisco"],
"Texas":["Houston","Dallas","Austin","San Antonio"],
"Florida":["Miami","Orlando","Tampa"],
"New York":["New York","Buffalo","Rochester"],
"Illinois":["Chicago","Springfield"]
}

REGIONS = {
"California":"West",
"Texas":"South",
"Florida":"South",
"New York":"East",
"Illinois":"Midwest"
}

CARRIERS = ["FedEx","UPS","DHL","USPS"]

# CATEGORY - BRAND MAPPING

CATEGORY_BRANDS = {

"Electronics":[
"Apple","Samsung","Sony","LG","Panasonic","Philips"
],

"Clothing":[
"Nike","Adidas"
],

"Groceries":[
"Nestle","PepsiCo"
],

"Beauty":[
"Procter & Gamble","Unilever"
],

"Home Appliances":[
"LG","Philips","Panasonic"
]

}

PRODUCT_CATALOG = {

"Electronics":[
"Smartphone","Laptop","Tablet",
"Smart TV","Bluetooth Headphones","Gaming Console"
],

"Home Appliances":[
"Microwave Oven","Air Fryer","Dishwasher",
"Vacuum Cleaner","Coffee Maker"
],

"Clothing":[
"Men T-Shirt","Women Jeans",
"Running Shoes","Hoodie","Winter Jacket"
],

"Groceries":[
"Organic Milk","Whole Wheat Bread",
"Greek Yogurt","Orange Juice","Brown Eggs"
],

"Beauty":[
"Shampoo","Conditioner",
"Face Moisturizer","Body Lotion","Sunscreen"
]

}

WAREHOUSES = [

("WH-001","Los Angeles","California"),
("WH-002","Dallas","Texas"),
("WH-003","Chicago","Illinois"),
("WH-004","New York","New York"),
("WH-005","Miami","Florida"),
("WH-006","Houston","Texas"),
("WH-007","San Francisco","California"),
("WH-008","Orlando","Florida"),
("WH-009","Austin","Texas"),
("WH-010","Buffalo","New York")

]

In [9]:
# GENERATE SUPPLIERS

supplier_rows = []
supplier_lookup = {}

supplier_id = 1

for category,brands in CATEGORY_BRANDS.items():

    for brand in brands:

        if brand not in supplier_lookup:

            sid = f"SUP-{supplier_id:03}"

            supplier_lookup[brand] = sid

            supplier_rows.append({
                "supplier_id":sid,
                "supplier_name":brand,
                "category":category,
                "country":"United States"
            })

            supplier_id+=1

suppliers_df = pd.DataFrame(supplier_rows)
suppliers_df.to_csv(f"{DATA_DIR}/suppliers.csv",index=False)

print(suppliers_df.head(10))
print("Suppliers generated")

  supplier_id supplier_name     category        country
0     SUP-001         Apple  Electronics  United States
1     SUP-002       Samsung  Electronics  United States
2     SUP-003          Sony  Electronics  United States
3     SUP-004            LG  Electronics  United States
4     SUP-005     Panasonic  Electronics  United States
5     SUP-006       Philips  Electronics  United States
6     SUP-007          Nike     Clothing  United States
7     SUP-008        Adidas     Clothing  United States
8     SUP-009        Nestle    Groceries  United States
9     SUP-010       PepsiCo    Groceries  United States
Suppliers generated


In [ ]:
# GENERATE PRODUCTS

products = []
pid = 1

for category,items in PRODUCT_CATALOG.items():

    brands = CATEGORY_BRANDS[category]

    for item in items:

        selected_brands = random.sample(brands,min(3,len(brands)))

        for brand in selected_brands:

            products.append({

                "product_id":f"PROD-{pid:05}",

                "product_name":f"{brand} {item}",

                "category":category,

                "brand":brand,

                "supplier_id":supplier_lookup[brand],

                "unit_price":round(random.uniform(10,800),2)

            })

            pid+=1

products_df = pd.DataFrame(products)
products_df.to_csv(f"{DATA_DIR}/products.csv",index=False)

print(products_df.head(10))
print("Products generated")

product_ids = products_df.product_id.tolist()

In [ ]:
# STORES

stores = []

for i in range(NUM_STORES):

    state = random.choice(list(STATES_CITIES.keys()))
    city = random.choice(STATES_CITIES[state])

    stores.append({

        "store_id":f"STORE-{i:04}",

        "store_name":f"{city} Retail Store {i}",

        "city":city,

        "state":state,

        "region":REGIONS[state],

        "store_open_date":datetime(2015,1,1) + timedelta(days=random.randint(0,3500))

    })

stores_df = pd.DataFrame(stores)
stores_df.to_csv(f"{DATA_DIR}/stores.csv",index=False)

print(stores_df.head(10))
print("Stores generated")

store_ids = stores_df.store_id.tolist()

In [15]:
# WAREHOUSES

warehouse_df = pd.DataFrame(
WAREHOUSES,
columns=["warehouse_id","city","state"]
)

warehouse_df.to_csv(f"{DATA_DIR}/warehouses.csv",index=False)

print("Warehouses generated")

Warehouses generated


In [16]:
# SEASONAL SALES WEIGHTS

CATEGORY_DEMAND_WEIGHT = {

"Electronics":1.4,
"Clothing":1.1,
"Groceries":1.6,
"Beauty":1.0,
"Home Appliances":0.8

}


In [ ]:
# INVENTORY SNAPSHOTS

for d in range(DAYS):

    date = START_DATE + timedelta(days=d)

    rows = []

    for wh in WAREHOUSES:

        sampled_products = np.random.choice(product_ids,INVENTORY_PRODUCTS_PER_WH)

        for p in sampled_products:

            base_qty = max(0,int(np.random.normal(200,60)))

            rows.append({

                "warehouse_id":wh[0],

                "product_id":p,

                "quantity_available":base_qty,

                "reorder_threshold":random.randint(30,80),

                "snapshot_date":date.strftime("%Y-%m-%d")

            })

    df = pd.DataFrame(rows)

    # introduce duplicates intentionally
    if random.random() < 0.2:
        df = pd.concat([df,df.sample(100)])

    df.to_csv(
        f"{DATA_DIR}/warehouse_inventory/inventory_{date.date()}.csv",
        index=False
    )

print(df.head(10))
print("Inventory snapshots generated")

In [ ]:
# SHIPMENTS

for d in range(DAYS):

    shipment_date = START_DATE + timedelta(days=d)

    records = []

    for _ in tqdm(range(SHIPMENTS_PER_DAY)):

        product = random.choice(product_ids)
        store = random.choice(store_ids)
        warehouse = random.choice(WAREHOUSES)[0]

        expected = shipment_date + timedelta(days=random.randint(1,5))

        # introduce delays
        actual = expected + timedelta(days=random.choice([0,0,0,1,2,3]))

        records.append({

            "shipment_id":str(uuid.uuid4()),

            "warehouse_id":warehouse,

            "store_id":store,

            "product_id":product,

            "quantity_shipped":random.randint(5,200),

            "shipment_date":shipment_date.isoformat(),

            "expected_delivery_date":expected.isoformat(),

            "actual_delivery_date":actual.isoformat(),

            "carrier":random.choice(CARRIERS)

        })

    with open(
        f"{DATA_DIR}/shipments/shipments_{shipment_date.date()}.json",
        "w"
    ) as f:

        json.dump(records,f)

print("Shipment logs generated")